<a href="https://colab.research.google.com/github/shubhamkamate2005-netizen/Loan_approval_prediction/blob/master/Telcom_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# ---------------------------------------------------------
# Step 1: Initialize Spark Session & Load Data
# ---------------------------------------------------------
print("Initializing Spark and loading data...")
spark = SparkSession.builder.appName("TelecomChurnPipeline").getOrCreate()

# Load CSV (Make sure the path matches where your file is stored)
df = spark.read.csv("churn_data_1M.csv", header=True, inferSchema=True)

# ---------------------------------------------------------
# Step 2: Data Cleaning
# ---------------------------------------------------------
# Drop unnecessary identifier columns
df = df.drop("Unnamed: 0", "customerID")

# TotalCharges comes in as a string due to empty spaces for new customers.
# Cast it to Double (spaces become nulls), then fill those nulls with 0.0
df = df.withColumn("TotalCharges", col("TotalCharges").cast("double"))
df = df.fillna({"TotalCharges": 0.0})

# ---------------------------------------------------------
# Step 3: Feature Engineering Setup
# ---------------------------------------------------------
# Separate categorical and numerical column names
categorical_cols = [f.name for f in df.schema.fields if f.dataType.typeName() == 'string' and f.name != 'Churn']
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

stages = [] # This list will hold all of our pipeline steps

# 1. Target Label Encoding (Convert Yes/No to 1.0/0.0)
label_indexer = StringIndexer(inputCol="Churn", outputCol="label")
stages += [label_indexer]

# 2. Categorical Encoding
for cat_col in categorical_cols:
    # First, convert strings to numerical indices
    indexer = StringIndexer(inputCol=cat_col, outputCol=cat_col + "_index", handleInvalid="keep")
    # Next, convert the indices into One-Hot Encoded vectors
    encoder = OneHotEncoder(inputCols=[indexer.getOutputCol()], outputCols=[cat_col + "_ohe"])
    stages += [indexer, encoder]

# 3. Vector Assembler (Combines all features into a single array column)
assembler_inputs = [c + "_ohe" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features")
stages += [assembler]

# 4. Standard Scaler (Scales the assembled vectors)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)
stages += [scaler]

# ---------------------------------------------------------
# Step 4: Model Definition
# ---------------------------------------------------------
# Add the Random Forest Classifier to the stages
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, seed=42)
stages += [rf]

# ---------------------------------------------------------
# Step 5: Build and Train the Pipeline
# ---------------------------------------------------------
# Combine all stages into a PySpark Pipeline
pipeline = Pipeline(stages=stages)

# Split data into 80% training and 20% testing
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print("Training the PySpark Random Forest model (this may take a moment)...")
model = pipeline.fit(train_data)

# ---------------------------------------------------------
# Step 6: Evaluation & Predictions
# ---------------------------------------------------------
print("Evaluating the model...")
predictions = model.transform(test_data)

# Evaluate Accuracy
acc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = acc_evaluator.evaluate(predictions)

# Evaluate Area Under ROC (Great metric for imbalanced classes like churn)
roc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = roc_evaluator.evaluate(predictions)

print(f"--- Results ---")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Area Under ROC (AUC): {auc:.4f}")

# Show a sample of the actual vs predicted labels
predictions.select("features", "label", "prediction", "probability").show(5, truncate=False)

Initializing Spark and loading data...
Training the PySpark Random Forest model (this may take a moment)...
Evaluating the model...
--- Results ---
Test Accuracy: 0.7310
Area Under ROC (AUC): 0.5046
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|features                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# This will create a folder inside your main Google Drive directory
model_path = "/content/drive/MyDrive/churn_pyspark_pipeline_model"

# Save the PySpark model directly to Google Drive
model.write().overwrite().save(model_path)

print(f"Model successfully saved to your Google Drive at: {model_path}")
model_path = "churn_pyspark_pipeline_model"
model.write().overwrite().save(model_path)

Mounted at /content/drive
Model successfully saved to your Google Drive at: /content/drive/MyDrive/churn_pyspark_pipeline_model
